# Tinjauan transform

Satu bagian per langkah transform. Di setiap bagian:
1. **Kode**: isi fungsi dari berkas `.py` (bukan salinan, jadi selalu yang terbaru).
2. **Hasil**: fungsi itu dijalankan pada data asli di `data_lake`.

Jalankan sel **Persiapan** dulu, lalu bagian yang ingin ditinjau.

In [9]:
# Persiapan
import glob
import importlib
import inspect
import json
import os
import sys
from pathlib import Path

# notebook ini ada di data-science/transform/; kerja dari data-science/
# supaya `from transform import ...` dan path data_lake/ jalan
akar = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "transform" / "parser.py").exists())
os.chdir(akar)
if str(akar) not in sys.path:
    sys.path.insert(0, str(akar))

import polars as pl
from IPython.display import HTML, display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer

from transform import parser

importlib.reload(parser)  # ambil versi .py terbaru tanpa restart kernel
pl.Config.set_tbl_rows(-1)
pl.Config.set_fmt_str_lengths(90)


def tampilkan_kode(objek):
    """Kode sumber modul/fungsi dengan warna tema gelap (github-dark)."""
    gaya = HtmlFormatter(style="github-dark", noclasses=True, nobackground=True)
    sumber = objek if isinstance(objek, str) else inspect.getsource(objek)
    isi = highlight(sumber, PythonLexer(), gaya)
    display(HTML(f'<div style="background:#0d1117;padding:12px;border-radius:6px">{isi}</div>'))

## Task 1A: berat (`transform/parser.py`)

Aturan: spec bagian 5.2. Fungsi yang dipakai transform nanti:
- `berat_gram(teks)`: berat dalam gram dari nama / variant / slug, atau `None`.
- `satuan_baris(nama, variant, banyak_opsi, usp)`: berat + sumber + jenis satuan + jumlah untuk satu baris. Produk **beberapa opsi** hanya membaca `variant` (A1).
- `pilih_berat(kandidat)`: sumber pertama yang punya berat (Blibli: `variant` → `nama`).
- `berat_dari_slug(url)`: cadangan Tokopedia kalau nama tanpa berat.
- `mangga_manalagi(nama)`: memuat mangga (atau ejaan lain: manga, mango, m4ngg4, …) **dan** manalagi.
- `jumlah_buah(nama)`, `jenis_satuan(nama, berat)`: `berat` / `jumlah` / `kemasan` / `lain`.

### Kode

In [10]:
tampilkan_kode(parser)

### Hasil: semua nama Mangga Manalagi unik (Blibli + Tokopedia, semua tanggal)

`cadangan` = `variant` (Blibli) atau URL tanpa query (Tokopedia).

In [11]:
baris = []
for berkas in glob.glob("data_lake/raw/blibli/date=*/*_page_*.json"):
    data = json.load(open(berkas, encoding="utf-8")).get("data") or {}
    for p in data.get("products") or []:
        if parser.mangga_manalagi(p.get("name")):
            baris.append({"sumber": "blibli", "nama": p["name"], "cadangan": p.get("variant") or "",
                           "banyak_opsi": bool(p.get("hasVariants")), "usp": p.get("uniqueSellingPoint")})
for berkas in glob.glob("data_lake*/raw/tokopedia/date=*/page_*.json"):
    try:
        produk = json.load(open(berkas, encoding="utf-8"))[0]["data"]["searchProductV5"]["data"]["products"]
    except (ValueError, KeyError, IndexError, TypeError):
        continue
    for p in produk:
        if parser.mangga_manalagi(p.get("name")):
            baris.append({"sumber": "tokopedia", "nama": p["name"], "cadangan": (p.get("url") or "").split("?")[0],
                           "banyak_opsi": False, "usp": None})


def hitung(b):
    if b["sumber"] == "blibli":
        s = parser.satuan_baris(b["nama"], b["cadangan"], b["banyak_opsi"], b["usp"])
        berat, sumber_berat, jenis = s.berat_gram, s.sumber_berat, s.jenis_satuan
    else:
        berat, sumber_berat = parser.pilih_berat([("nama", b["nama"])])
        if berat is None:
            berat = parser.berat_dari_slug(b["cadangan"])
            sumber_berat = "url" if berat is not None else None
        jenis = parser.jenis_satuan(b["nama"], berat)
    return {**{k: v for k, v in b.items() if k != "usp"},
            "berat_gram": berat, "sumber_berat": sumber_berat, "jenis_satuan": jenis}


berat = pl.DataFrame([hitung(b) for b in baris]).unique().sort("sumber", "jenis_satuan", "nama")
display(berat.group_by("sumber", "jenis_satuan").len().sort("sumber", "jenis_satuan"))
berat

sumber,jenis_satuan,len
str,str,u32
"""blibli""","""berat""",8
"""tokopedia""","""berat""",34
"""tokopedia""","""lain""",7


sumber,nama,cadangan,banyak_opsi,berat_gram,sumber_berat,jenis_satuan
str,str,str,bool,i64,str,str
"""blibli""","""Buah Mangga Manalagi segar""","""1/2 kg""",true,500,"""variant""","""berat"""
"""blibli""","""MANGGA MANALAGI 170 - 180 G""","""""",false,175,"""nama""","""berat"""
"""blibli""","""MANGGA MANALAGI 550-570 GR""","""""",false,560,"""nama""","""berat"""
"""blibli""","""Mangga Manalagi""","""1 kg""",false,1000,"""variant""","""berat"""
"""blibli""","""Mangga Manalagi 1 kg ( 0.9 - 1.1 kg / pack)""","""""",false,1000,"""nama""","""berat"""
"""blibli""","""Mangga Manalagi 500 gram ( 400 - 500 gram / pack)""","""""",false,500,"""nama""","""berat"""
"""blibli""","""Mangga Manalagi Buah Segar Ibaza fruitsBOX - 1 kg Buah Mangga Manalagi""","""""",false,1000,"""nama""","""berat"""
"""blibli""","""Mangga manalagi segar manis 1/2 kg""","""""",false,500,"""nama""","""berat"""
"""tokopedia""","""1 Kg Mangga MANALAGI MADU Manis Legit dikirim Mengkal""","""https://www.tokopedia.com/murahmurahkediri3/1-kg-mangga-manalagi-madu-manis-legit-dikirim-…",false,1000,"""nama""","""berat"""


### Hasil per opsi: produk mangga Blibli yang punya varian (tanggal terbaru)

Satu nama, beberapa opsi (`item_sku`), harga berbeda. **Keputusan A1 (8 Nov):** berat/jumlah
hanya dari `variant` opsi itu, **tidak pernah dipinjam dari nama** ("Buy 1 Get 1" → `lain`). `harga_per_kg` = pratinjau Task 7.
Data opsi baru ada sejak 7–8 Nov.

In [12]:
opsi = []
for berkas in sorted(glob.glob("data_lake/raw/blibli/date=*/varian/ps--*.json")):
    ringkasan = json.load(open(berkas, encoding="utf-8"))["data"]
    if "mangga" not in (ringkasan.get("name") or "").lower():
        continue
    for o in ringkasan.get("options") or []:
        kode = o["id"].split("_")[0]                      # "is--SKU_PP-123" -> "is--SKU"
        if o.get("selected"):
            harga = ringkasan["price"]["offered"]         # opsi default: harga ada di ps--
        else:
            berkas_opsi = Path(berkas).parent / f"{kode}.json"
            harga = (json.load(open(berkas_opsi, encoding="utf-8"))["data"]["price"]["offered"]
                     if berkas_opsi.exists() else None)
        variant = o["attributes"][0]["value"] if o.get("attributes") else None
        s = parser.satuan_baris(ringkasan["name"], variant, banyak_opsi=True)
        opsi.append({
            "tanggal": berkas.split("date=")[1][:10],
            "nama": ringkasan["name"],
            "item_sku": kode.removeprefix("is--"),
            "opsi_default": bool(o.get("selected")),
            "variant": variant,
            "harga": harga,
            "berat_gram": s.berat_gram,
            "sumber_berat": s.sumber_berat,
            "jenis_satuan": s.jenis_satuan,
            "jumlah_buah": s.jumlah_buah,
            "harga_per_kg": round(harga / s.berat_gram * 1000) if harga and s.berat_gram else None,
            "harga_per_buah": round(harga / s.jumlah_buah) if harga and s.jumlah_buah else None,
        })

opsi = pl.DataFrame(opsi)
opsi.filter(pl.col("tanggal") == pl.col("tanggal").max()).sort("nama", "berat_gram")

tanggal,nama,item_sku,opsi_default,variant,harga,berat_gram,sumber_berat,jenis_satuan,jumlah_buah,harga_per_kg,harga_per_buah
str,str,str,bool,str,f64,i64,str,str,null,i64,null
"""2026-11-08""","""BUAH MANGGA HARUM MANIS SUPER PREMIUM QUALITY""","""BUS-70370-00021-00001""",true,"""800 GR""",30000.0,800,"""variant""","""berat""",null,37500,null
"""2026-11-08""","""BUAH MANGGA HARUM MANIS SUPER PREMIUM QUALITY""","""BUS-70370-00021-00002""",false,"""900 GR""",33000.0,900,"""variant""","""berat""",null,36667,null
"""2026-11-08""","""BUAH MANGGA HARUM MANIS SUPER PREMIUM QUALITY""","""BUS-70370-00021-00003""",false,"""1.000 GR""",36000.0,1000,"""variant""","""berat""",null,36000,null
"""2026-11-08""","""BUAH MANGGA HARUM MANIS SUPER PREMIUM QUALITY""","""BUS-70370-00021-00004""",false,"""PAKET 2 KG""",70000.0,2000,"""variant""","""berat""",null,35000,null
"""2026-11-08""","""Buah Mangga Indramayu""","""DOD-70072-00729-00002""",true,"""1/2 kg""",13500.0,500,"""variant""","""berat""",null,27000,null
"""2026-11-08""","""Buah Mangga Indramayu""","""DOD-70072-00729-00001""",false,"""1 kg""",20000.0,1000,"""variant""","""berat""",null,20000,null
"""2026-11-08""","""Buah Mangga Kweni""","""DOD-70072-00873-00002""",true,"""500g""",13000.0,500,"""variant""","""berat""",null,26000,null
"""2026-11-08""","""Buah Mangga Kweni""","""DOD-70072-00873-00001""",false,"""1000g""",25000.0,1000,"""variant""","""berat""",null,25000,null
"""2026-11-08""","""Buah Mangga Manalagi segar""","""DOD-70072-00076-00001""",true,"""1/2 kg""",13000.0,500,"""variant""","""berat""",null,26000,null


### Yang perlu ditinjau (3 pertanyaan, ±5 menit)

1. **Tanpa berat:** benarkah tidak ada berat tertulis di nama-nama ini?
2. **Nama sulit** (rentang, pecahan, garis miring): benarkah `berat_gram`-nya?
3. **Opsi yang mencurigakan:** opsi yang `harga_per_kg`-nya beda > 30% dari opsi lain di produk yang sama
   (tanda berat salah). Tabel kosong = tidak ada yang mencurigakan.

In [ ]:
display(berat.filter(pl.col("jenis_satuan") != "berat").select("sumber", "nama"))
display(berat.filter(pl.col("nama").str.contains(r"\d\s*-\s*\d|/")).select("sumber", "nama", "berat_gram"))
terbaru = opsi.filter(pl.col("tanggal") == pl.col("tanggal").max())
terbaru.with_columns(median_produk=pl.col("harga_per_kg").median().over("nama")).filter(
    (pl.col("harga_per_kg") / pl.col("median_produk") - 1).abs() > 0.3
).select("nama", "variant", "harga", "berat_gram", "harga_per_kg", "median_produk")

### Coba nama sendiri

Ganti teks di bawah lalu jalankan. Produk **tanpa opsi**: kosongkan `contoh_variant` (`""`).

In [13]:
contoh_nama = "Mangga Manalagi 900g - 1 Kg"
contoh_variant = "1 kg"   # "" kalau produk tanpa opsi

berat_contoh, sumber_contoh = parser.pilih_berat([("variant", contoh_variant), ("nama", contoh_nama)])
berat_contoh, sumber_contoh, parser.jenis_satuan(contoh_nama, berat_contoh)

(1000, 'variant', 'berat')

## Task 1B: jenis buah, olahan, asal, terjual, region (`transform/parser.py`)

- `jenis_buah(nama)`: kata kunci → `mangga`, `apel`, …, `lainnya`. Mangga ikut aturan salah ketik.
- `olahan(nama)`: penanda produk olahan (tanpa `sale` dan `pohon`, keputusan 8 Nov).
- `asal_dari_nama(nama)`: `impor` / `lokal` / `tidak_diketahui` (untuk Tokopedia).
- `terjual_dari_label(label)`, `ember_terjual(n)`: angka terjual Tokopedia / Blibli disetarakan.
- `region(kota)`: `jabodetabek` / `jawa_lain` / `luar_jawa` / `tidak_diketahui`.
- Multipack (`berat_dikali`): "8x125GR" → 1000 g, dicatat di karantina nanti (cek H).

### Kode

In [ ]:
sumber = inspect.getsource(parser)
tampilkan_kode(sumber[sumber.index("# ---- Task 1B ----"):])

### Hasil (tanggal terbaru, Blibli + Tokopedia)

In [ ]:
tanggal_baru = max(Path(f).name for f in glob.glob("data_lake/raw/blibli/date=*"))
produk = []
for berkas in glob.glob(f"data_lake/raw/blibli/{tanggal_baru}/*_page_*.json"):
    label_asal = "lokal" if "BU-1000190" in berkas else "impor"
    for p in (json.load(open(berkas, encoding="utf-8")).get("data") or {}).get("products") or []:
        produk.append({"sumber": "blibli", "nama": p["name"], "kota": p.get("location"),
                       "label_asal": label_asal, "label_terjual": None})
for berkas in glob.glob(f"data_lake/raw/tokopedia/{tanggal_baru}/page_*.json"):
    try:
        isi = json.load(open(berkas, encoding="utf-8"))[0]["data"]["searchProductV5"]["data"]["products"]
    except (ValueError, KeyError, IndexError, TypeError):
        continue
    for p in isi:
        label = next((l.get("title") for l in p.get("labelGroups") or []
                      if l.get("position") == "ri_product_credibility"), None)
        produk.append({"sumber": "tokopedia", "nama": p["name"], "kota": p["shop"].get("city"),
                       "label_asal": None, "label_terjual": label})

lain = pl.DataFrame(produk, infer_schema_length=None).with_columns(
    jenis_buah=pl.col("nama").map_elements(parser.jenis_buah, return_dtype=pl.Utf8),
    olahan=pl.col("nama").map_elements(parser.olahan, return_dtype=pl.Boolean),
    asal=pl.col("nama").map_elements(parser.asal_dari_nama, return_dtype=pl.Utf8),
    region=pl.col("kota").map_elements(parser.region, return_dtype=pl.Utf8, skip_nulls=False),
    terjual=pl.col("label_terjual").map_elements(parser.terjual_dari_label, return_dtype=pl.Int64),
    berat_gram=pl.col("nama").map_elements(parser.berat_gram, return_dtype=pl.Int64),
    berat_dikali=pl.col("nama").map_elements(parser.berat_dikali, return_dtype=pl.Boolean),
)
print(tanggal_baru, lain.height, "baris")
display(lain.group_by("jenis_buah").len().sort("len", descending=True))
display(lain.group_by("region").len().sort("len", descending=True))

### Yang perlu ditinjau

1. **Olahan:** apakah baris-baris ini memang bukan buah utuh?
2. **Asal (Blibli punya label asli):** baris yang aturannya salah.
3. **Multipack:** apakah hasil perkaliannya benar?
4. **Kota luar Jawa:** adakah kota Jawa yang salah masuk ke sini?

In [ ]:
display(lain.filter(pl.col("olahan")).select("sumber", "nama").unique().sort("nama").head(40))
display(lain.filter(pl.col("label_asal").is_not_null() & (pl.col("asal") != "tidak_diketahui")
                    & (pl.col("asal") != pl.col("label_asal"))).select("nama", "label_asal", "asal").unique())
display(lain.filter(pl.col("berat_dikali")).select("sumber", "nama", "berat_gram").unique())
lain.filter(pl.col("region") == "luar_jawa").select("kota").unique().sort("kota")